# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.03 · Revisión LLM dirigida

Prioriza desacuerdos, baja confianza, contexto y clases minoritarias sin tratarlos como verdad humana.

La selección por incertidumbre pertenece a la familia de aprendizaje activo [1], mientras que el balance puede aumentar la atención sobre clases raras [2]. En lenguaje abusivo, el contexto conversacional puede cambiar la interpretación del fragmento [3]. Como una sugerencia LLM puede influir en la decisión humana [4], el ordenamiento, el umbral 0.8 y la posibilidad de ocultar la sugerencia se tratan como decisiones locales que deben auditarse.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Selección reproducible

In [ ]:
from tqdm.auto import tqdm
from moderacion_peru.io import read_jsonl
SOURCE=ROOT/'datos/etiquetado/local/ollama_qwen35_4b_v2.jsonl'
review=[]
scanned=0
if SOURCE.exists():
    for row in tqdm(read_jsonl(SOURCE),desc='Construyendo cola dirigida',unit='anotación'):
        scanned+=1
        if row.get('needs_review') or row.get('score_confianza',1)<0.8:
            review.append(row)
review.sort(key=lambda row:row['chunk_id'])
show_summary('Cola de revisión dirigida',{'etiquetados':scanned,'requieren_revisión':len(review),'umbral_confianza':0.8},tone='warning' if review else 'success')

## Siguiente paso

In [ ]:
show_callout('Siguiente paso', 'La revisión puede usar otro modelo o pasar directamente a 02_04 para validación humana.', tone='neutral')

## Referencias

[1] B. Settles, "Active Learning Literature Survey," Univ. Wisconsin–Madison, Computer Sciences Tech. Rep. 1648, 2009. [Online]. Available: https://minds.wisconsin.edu/handle/1793/60660

[2] Y. Fairstein, O. Kalinsky, Z. Karnin, et al., "Class Balancing for Efficient Active Learning in Imbalanced Datasets," in Proc. 18th Linguistic Annotation Workshop, 2024, pp. 77–86, doi: 10.18653/v1/2024.law-1.8.

[3] T. Bourgeade, Z. Li, F. Benamara, et al., "Humans Need Context, What about Machines? Investigating Conversational Context in Abusive Language Detection," in Proc. LREC-COLING, 2024, pp. 8438–8452. [Online]. Available: https://aclanthology.org/2024.lrec-main.740/

[4] A. S. Choi, S. S. Akter, J. P. Singh, et al., "The LLM Effect: Are Humans Truly Using LLMs, or Are They Being Influenced By Them Instead?" in Proc. EMNLP, 2024, pp. 22032–22054, doi: 10.18653/v1/2024.emnlp-main.1230.